# Possession Starter Analysis

How many points does a team score on a possession depending on **how they got the ball**?

We use `prev_poss_ender` from `possessions_enriched` as the "possession starter" — it captures what ended the *previous* possession (and thus how the current team received the ball):

| prev_poss_ender | Meaning |
|---|---|
| `made_fg` | Got ball after opponent made a field goal |
| `made_ft` | Got ball after opponent made last free throw |
| `fga_def_rebound` | Defensive rebound of opponent's missed FG |
| `fta_def_rebound` | Defensive rebound of opponent's missed FT |
| `live_ball_turnover` | Steal — live ball |
| `dead_ball_turnover` | Turnover, out of bounds, etc. |
| `dead_ball_rebound` | Jump ball / out of bounds rebound |
| `start_of_period` | First possession of a half |

Points are computed by summing `scoreValue` (filtered to `scoringPlay == True`) from `plays` for each possession's play rows.

> **Data note:** possessions regenerated with `fix_possessions.py` v6, which fixes (a) phantom possession splits caused by substitutions mid-FT-sequence, and (b) steal plays being assigned to the wrong possession (causing live ball turnovers to be misclassified as dead ball).

In [ ]:
import sys, os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── locate cbbd_data ─────────────────────────────────────────────────────────
# __vsc_ipynb_file__ is set by VSCode; fall back to walking up from cwd
try:
    _nb_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _nb_dir = os.path.abspath('')
    while _nb_dir != os.path.dirname(_nb_dir):
        if os.path.isdir(os.path.join(_nb_dir, 'cbbd_data')):
            break
        _nb_dir = os.path.dirname(_nb_dir)

CBBD_DIR = os.path.join(_nb_dir, 'cbbd_data')
print('cbbd_data dir:', CBBD_DIR)
assert os.path.isdir(CBBD_DIR), f'cbbd_data not found at {CBBD_DIR}'

In [ ]:
# ── load all possessions_enriched ────────────────────────────────────────────
pe_paths = sorted(glob.glob(os.path.join(CBBD_DIR, 'possessions_enriched', '*.csv')))
pe = pd.concat([pd.read_csv(p) for p in pe_paths], ignore_index=True)
pe = pe.drop_duplicates(subset=['gameId', 'possession_id', 'possession_team'])
print(f'possessions_enriched: {len(pe):,} rows')
print('prev_poss_ender distribution:')
print(pe['prev_poss_ender'].value_counts())

In [ ]:
# ── load all raw possessions (play-level, has possession_id tagged) ───────────
pr_paths = sorted(glob.glob(os.path.join(CBBD_DIR, 'possessions', '*.csv')))
pr = pd.concat([pd.read_csv(p) for p in pr_paths], ignore_index=True)
pr = pr.drop_duplicates(subset=['play_id', 'gameId', 'possession_team'])
print(f'possessions (raw): {len(pr):,} rows')

# ── load all plays (has scoreValue + playType for classification) ──────────────
pl_paths = sorted(glob.glob(os.path.join(CBBD_DIR, 'plays', '*.csv')))
pl = pd.concat(
    [pd.read_csv(p, usecols=['id', 'scoreValue', 'scoringPlay', 'playType', 'gameId']) for p in pl_paths],
    ignore_index=True
)
pl = pl.drop_duplicates(subset=['id'])
print(f'plays: {len(pl):,} rows')

# ── load shots (shot zone info for DREB + DBR sub-classification) ─────────────
sh_paths = sorted(glob.glob(os.path.join(CBBD_DIR, 'shots', '*.csv')))
sh = pd.concat(
    [pd.read_csv(p, usecols=['play_id', 'shot_range', 'is_three', 'distance', 'y', 'gameId'])
     for p in sh_paths],
    ignore_index=True
)
sh = sh.drop_duplicates(subset=['play_id'])
print(f'shots: {len(sh):,} rows')

In [ ]:
# ── shared helpers ────────────────────────────────────────────────────────────
_SKIP_PT  = {'Substitution','Official TV Timeout','OfficialTVTimeOut','ShortTimeOut','RegularTimeOut',''}
_FG_TYPES = {'JumpShot','LayUpShot','DunkShot','TipShot'}

# Join playType onto raw possession rows — used by both classification cells
pr_typed = pr.merge(
    pl[['id', 'playType', 'gameId']].rename(columns={'id': 'play_id'}),
    on=['play_id', 'gameId'], how='left'
)

def _prev_team_lookup(ender_name):
    """Return prev-possession team for all possessions with prev_poss_ender == ender_name."""
    sub = pe[pe['prev_poss_ender'] == ender_name][
        ['gameId', 'possession_id', 'possession_team']].copy()
    sub['prev_poss_id'] = sub['possession_id'] - 1
    sub = sub.merge(
        pe[['gameId', 'possession_id', 'possession_team']].rename(
            columns={'possession_id': 'prev_poss_id', 'possession_team': 'prev_team'}),
        on=['gameId', 'prev_poss_id'], how='left'
    )
    return sub

def _prev_plays(prev_lookup):
    """All play rows for the previous possessions referenced by prev_lookup."""
    return pr_typed.merge(
        prev_lookup[['gameId', 'prev_poss_id', 'prev_team']].drop_duplicates(),
        left_on=['gameId', 'possession_id', 'possession_team'],
        right_on=['gameId', 'prev_poss_id', 'prev_team'],
        how='inner'
    )

# ── classify dead_ball_rebound into sub-types ─────────────────────────────────
# Trigger = play right before the Dead Ball Rebound in the previous possession
dbr_pe   = _prev_team_lookup('dead_ball_rebound')
dbr_prev = _prev_plays(dbr_pe)

def _get_dbr_trigger(group):
    g = group[group['playType'].notna() & ~group['playType'].isin(_SKIP_PT)]
    g = g.sort_values('play_id', ascending=False)
    if len(g) < 2:
        return pd.Series({'trigger_play_id': None, 'trigger_type': 'only_one_play'})
    return pd.Series({'trigger_play_id': g.iloc[1]['play_id'],
                      'trigger_type':    g.iloc[1]['playType']})

dbr_triggers = (
    dbr_prev
    .groupby(['gameId', 'prev_poss_id', 'prev_team'], group_keys=False)
    .apply(_get_dbr_trigger, include_groups=False)
    .reset_index()
).merge(sh[['play_id', 'is_three', 'gameId']].rename(columns={'play_id': 'trigger_play_id'}),
        on=['trigger_play_id', 'gameId'], how='left')

def _classify_dbr(row):
    pt = row['trigger_type']
    if pt in ('LayUpShot', 'DunkShot', 'TipShot'): return 'dbr_oob_rim'
    if pt == 'JumpShot':  return 'dbr_oob_3' if row.get('is_three') == True else 'dbr_oob_2'
    if pt == 'MadeFreeThrow': return 'dbr_oob_ft'
    if pt == 'Block Shot':    return 'dbr_blocked'
    return 'dbr_other'

dbr_triggers['dbr_sub'] = dbr_triggers.apply(_classify_dbr, axis=1)
dbr_lookup = dbr_pe.merge(
    dbr_triggers[['gameId', 'prev_poss_id', 'prev_team', 'dbr_sub']],
    on=['gameId', 'prev_poss_id', 'prev_team'], how='left'
)[['gameId', 'possession_id', 'possession_team', 'dbr_sub']].copy()
dbr_lookup['dbr_sub'] = dbr_lookup['dbr_sub'].fillna('dbr_other')

print('DBR sub-types:')
print(dbr_lookup['dbr_sub'].value_counts())

In [ ]:
# ── classify fga_def_rebound into sub-types ───────────────────────────────────
dreb_pe   = _prev_team_lookup('fga_def_rebound')
dreb_prev = _prev_plays(dreb_pe)

def _get_dreb_shot(group):
    """Walk backward through prev possession to find last FG + detect if it was blocked."""
    g = group[group['playType'].notna()].sort_values('play_id', ascending=False)
    blocked    = False
    fg_play_id = None
    for _, row in g.iterrows():
        pt = row['playType']
        if pt == 'Block Shot':
            blocked = True
        elif pt in _FG_TYPES:
            fg_play_id = row['play_id']
            break
    return pd.Series({'fg_play_id': fg_play_id, 'blocked': blocked})

dreb_shots = (
    dreb_prev
    .groupby(['gameId', 'prev_poss_id', 'prev_team'], group_keys=False)
    .apply(_get_dreb_shot, include_groups=False)
    .reset_index()
).merge(
    sh[['play_id', 'shot_range', 'is_three', 'distance', 'y', 'gameId']]
      .rename(columns={'play_id': 'fg_play_id'}),
    on=['fg_play_id', 'gameId'], how='left'
)

def _classify_dreb(row):
    sr   = row.get('shot_range')
    dist = row.get('distance')
    y    = row.get('y')
    pfx  = 'dreb_block' if row['blocked'] else 'dreb'
    if sr == 'rim':
        return f'{pfx}_ra'
    elif sr == 'jumper':
        return f'{pfx}_paint' if (pd.notna(dist) and dist <= 10) else f'{pfx}_mid'
    elif sr == 'three_pointer' or row.get('is_three') == True:
        return f'{pfx}_c3' if (pd.notna(y) and y < 95) else f'{pfx}_nc3'
    return f'{pfx}_other'

dreb_shots['dreb_sub'] = dreb_shots.apply(_classify_dreb, axis=1)
dreb_lookup = dreb_pe.merge(
    dreb_shots[['gameId', 'prev_poss_id', 'prev_team', 'dreb_sub']],
    on=['gameId', 'prev_poss_id', 'prev_team'], how='left'
)[['gameId', 'possession_id', 'possession_team', 'dreb_sub']].copy()
dreb_lookup['dreb_sub'] = dreb_lookup['dreb_sub'].fillna('dreb_other')

print('DREB sub-types:')
print(dreb_lookup['dreb_sub'].value_counts())

In [ ]:
# ── compute points scored per possession ──────────────────────────────────────
pr_scored = pr.merge(
    pl[['id', 'scoreValue', 'scoringPlay', 'gameId']].rename(columns={'id': 'play_id'}),
    on=['play_id', 'gameId'], how='left'
)
poss_pts = (
    pr_scored[pr_scored['scoringPlay'] == True]
    .groupby(['gameId', 'possession_id', 'possession_team'], as_index=False)['scoreValue']
    .sum()
    .rename(columns={'scoreValue': 'points'})
)
print(f'Scored possessions: {len(poss_pts):,}  (zero-point possessions will get 0 on merge)')

In [ ]:
# ── merge points onto possessions_enriched ────────────────────────────────────
df = pe.merge(poss_pts, on=['gameId', 'possession_id', 'possession_team'], how='left')
df['points'] = df['points'].fillna(0)
df = df[df['possession_team'].notna()].copy()

# Merge both sub-type lookups and build prev_poss_ender_detail
df = df.merge(dbr_lookup,  on=['gameId', 'possession_id', 'possession_team'], how='left')
df = df.merge(dreb_lookup, on=['gameId', 'possession_id', 'possession_team'], how='left')

df['prev_poss_ender_detail'] = df.apply(
    lambda r: (r['dbr_sub']  if r['prev_poss_ender'] == 'dead_ball_rebound'
          else r['dreb_sub'] if r['prev_poss_ender'] == 'fga_def_rebound'
          else r['prev_poss_ender']),
    axis=1
)

print(f'Final df: {len(df):,} possessions')
print(f'Season avg PPP: {df["points"].mean():.3f}')
print('\nprev_poss_ender_detail distribution:')
print(df['prev_poss_ender_detail'].value_counts())

In [ ]:
# ── aggregate by prev_poss_ender_detail ───────────────────────────────────────
EXCLUDE_ENDERS = {'start_of_period', 'end_period'}
analysis = df[~df['prev_poss_ender_detail'].isin(EXCLUDE_ENDERS)].copy()

summary = (
    analysis
    .groupby('prev_poss_ender_detail')
    .agg(
        possessions=('points', 'count'),
        total_points=('points', 'sum'),
        ppp=('points', 'mean'),
        pct_0pts=('points', lambda x: (x == 0).mean()),
        pct_2pts=('points', lambda x: (x == 2).mean()),
        pct_3pts=('points', lambda x: (x == 3).mean()),
    )
    .reset_index()
    .sort_values('ppp', ascending=False)
)

LABEL_MAP = {
    # Turnovers
    'live_ball_turnover':  'Live Ball TO (Steal)',
    'dead_ball_turnover':  'Dead Ball TO',
    # Standard
    'made_fg':             'After Opp Made FG',
    'made_ft':             'After Opp Made FT',
    'fta_def_rebound':     'Def Reb (missed FT)',
    # DREB by shot zone — unblocked
    'dreb_ra':             'DREB: Restricted Area',
    'dreb_paint':          'DREB: 2pt Paint',
    'dreb_mid':            'DREB: 2pt Outside Paint',
    'dreb_c3':             'DREB: Corner 3',
    'dreb_nc3':            'DREB: Non-Corner 3',
    # DREB by shot zone — blocked
    'dreb_block_ra':       'Block DREB: Restricted Area',
    'dreb_block_paint':    'Block DREB: 2pt Paint',
    'dreb_block_mid':      'Block DREB: 2pt Outside Paint',
    'dreb_block_c3':       'Block DREB: Corner 3',
    'dreb_block_nc3':      'Block DREB: Non-Corner 3',
    'dreb_block_other':    'Block DREB: Other',
    'dreb_other':          'DREB: Other',
    # OOB (dead ball rebound sub-types)
    'dbr_oob_3':           'OOB: Opp Missed 3',
    'dbr_oob_2':           'OOB: Opp Missed 2',
    'dbr_oob_rim':         'OOB: Opp Missed Rim',
    'dbr_oob_ft':          'OOB: Own Missed FT',
    'dbr_blocked':         'OOB: Block OOB',
    'dbr_other':           'OOB: Other',
}
summary['label'] = summary['prev_poss_ender_detail'].map(LABEL_MAP).fillna(summary['prev_poss_ender_detail'])

print(summary[['label', 'possessions', 'ppp', 'pct_0pts', 'pct_2pts', 'pct_3pts']]
      .to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# ── Chart 1: PPP by possession starter ───────────────────────────────────────
analysis['label'] = analysis['prev_poss_ender_detail'].map(LABEL_MAP).fillna(analysis['prev_poss_ender_detail'])

# Color by category
def _row_color(lbl):
    if 'Steal' in lbl or 'Live Ball' in lbl: return '#E53935'
    if 'Dead Ball TO' in lbl:                return '#FB8C00'
    if 'Block DREB' in lbl:                  return '#7B1FA2'
    if 'DREB' in lbl:                        return '#1E88E5'
    if 'OOB' in lbl:                         return '#43A047'
    return '#757575'

colors = [_row_color(lbl) for lbl in summary['label']]

fig, ax = plt.subplots(figsize=(11, max(8, len(summary) * 0.38)))
bars = ax.barh(summary['label'], summary['ppp'], color=colors, edgecolor='white', height=0.65)
for bar, val in zip(bars, summary['ppp']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8)
ax.axvline(analysis['points'].mean(), color='black', linestyle='--', linewidth=1,
           label=f'Season avg ({analysis["points"].mean():.3f})')
ax.set_xlabel('Points Per Possession (PPP)', fontsize=11)
ax.set_title('PPP by How the Possession Started', fontsize=13, fontweight='bold')
ax.set_xlim(0, summary['ppp'].max() * 1.15)
ax.invert_yaxis()
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: Scoring distribution by possession starter ──────────────────────
# Stacked horizontal bar: % of possessions scoring 0 / 2 / 3 / other pts
plot_df = summary.sort_values('ppp', ascending=True)   # bottom = lowest PPP

pct_0 = plot_df['pct_0pts'].values * 100
pct_2 = plot_df['pct_2pts'].values * 100
pct_3 = plot_df['pct_3pts'].values * 100
pct_x = 100 - pct_0 - pct_2 - pct_3    # 1-pt FT, 4-pt plays, etc.

labels = plot_df['label'].values

fig, ax = plt.subplots(figsize=(11, max(8, len(plot_df) * 0.38)))
left = np.zeros(len(plot_df))
for vals, seg_label, color in [
    (pct_0, '0 pts (scoreless)', '#EF5350'),
    (pct_x, '1 or 4+ pts',       '#FFCA28'),
    (pct_2, '2 pts',              '#42A5F5'),
    (pct_3, '3 pts',              '#66BB6A'),
]:
    ax.barh(labels, vals, left=left, label=seg_label, color=color,
            edgecolor='white', height=0.65)
    left = left + vals

ax.set_xlabel('% of Possessions', fontsize=11)
ax.set_title('Scoring Distribution by Possession Starter', fontsize=13, fontweight='bold')
ax.set_xlim(0, 100)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.invert_yaxis()
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Full summary table ─────────────────────────────────────────────────────────
out = summary[['label', 'possessions', 'ppp', 'pct_0pts', 'pct_2pts', 'pct_3pts']].copy()
out.columns = ['Possession Starter', 'Possessions', 'PPP', '% Scoreless', '% 2-pt Poss', '% 3-pt Poss']
out['PPP']         = out['PPP'].round(3)
out['% Scoreless'] = (out['% Scoreless'] * 100).round(1)
out['% 2-pt Poss'] = (out['% 2-pt Poss'] * 100).round(1)
out['% 3-pt Poss'] = (out['% 3-pt Poss'] * 100).round(1)
display(out.reset_index(drop=True))